# HDBSCAN Clustering for RBD sequences, MSA
---
For a baseline comparison to ESM embedded sequences, we will use multiple sequence alignment, and then use one hot to get a numerical representation. Then use UMAP/tSNE+HDBSCAN just like in the ESM embedded sequence clustering.

In [4]:
version_dir = "rbd_ado_msa"
version_prefix = "RBD.ADO.MSA_one_hot-embedded"

In [3]:
import os
import pandas as pd

data_dir = "/panfs/biopan03/prime_ml/prime/data/rbd/old"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/rbd/parquets"

parquet_file = os.path.join(data_dir, "spikeprot0528.clean.uniq.noX.RBD.metadata.variants.ADO.CLS-embedded.parquet")
embedded_ado_df = pd.read_parquet(parquet_file, engine='fastparquet')
seq_ids = set(embedded_ado_df["seq_id"])
print(len(seq_ids))

291539


We load in the MSA data + filter seq_ids:

In [ ]:
# (Records only)
import numpy as np
from Bio import AlignIO

alignment = AlignIO.read("/panfs/biopan03/prime_ml/prime/notebooks/phylogenetic_analysis/aligned_ADO_seq.afasta", "fasta")

# Filter records by ID
filtered_alignment = [record for record in alignment if record.id in seq_ids]

# Extract (ID, description) for each sequence
metadata = [(record.id, str(record.seq), record.description.split("|")[1].strip(), record.description.split("|")[2].strip()) for record in filtered_alignment]

# Convert to DataFrame
msa_df = pd.DataFrame(metadata, columns=["seq_id", "seq", "variant", "pango lineage"])

# Save as Parquet
msa_df.to_parquet(os.path.join(parquet_dir, version_dir, "RBD.ADO.MSA.parquet"), index=False)

One-Hot

In [8]:
aas = "ACDEFGHIKLMNPQRSTVWY-"  # include gap
aa_to_idx = {aa: i for i, aa in enumerate(aas)}

n_seq = len(filtered_alignment)
align_len = alignment.get_alignment_length()    # seq length should not change post filtering
X = np.zeros((n_seq, align_len * len(aas)), dtype=np.int8)

for i, record in enumerate(filtered_alignment):
    for j, aa in enumerate(str(record.seq)):
        X[i, j * len(aas) + aa_to_idx.get(aa, 20)] = 1  # unknown -> gap index

np.savez_compressed(os.path.join(parquet_dir, version_dir, "RBD.ADO.MSA_one_hot-embedded.compressed.npz"), X=X)

In [16]:
one_hot_msa_df = msa_df.copy()
one_hot_msa_df["embedding"] = list(X)
one_hot_msa_df.to_parquet(os.path.join(parquet_dir, version_dir, "RBD.ADO.MSA_one_hot-embedded.parquet"), index=False)
display(one_hot_msa_df)

,seq_id,seq,variant,pango lineage,embedding
0,EPI_ISL_16604644,RVQ--PTESIVRFPN-ITNLCP-------------FDEV-FNAT--...,Omicron,DY.1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
1,EPI_ISL_14904955,RVQ--PTESIVRFPNI-TNLCP-------------FDEVF-NAT--...,Omicron,BA.5.5,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
2,EPI_ISL_14904971,RVQ--PTESIVRFPNI-TNLCP-------------FDEVF-NAT--...,Omicron,BA.5.6,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
3,EPI_ISL_14906096,RVQ--PTESIVRFPNI-TNLCP-------------FDEVF-NAT--...,Omicron,BA.5.1.10,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
4,EPI_ISL_14906040,RVQ--PTESIVRFPNI-TNLCP-------------FDEVF-NAT--...,Omicron,BA.5.2.9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
...,...,...,...,...,...
291534,EPI_ISL_9835815,RVQ--PTESIVRFPNI-TNLCP-------------FDEVF-NAT--...,Omicron,BA.1,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
291535,EPI_ISL_412977,RIL--PSTEVVRFPN-ITNFCP-------------FDKV-FNAT--...,Delta,B,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
291536,EPI_ISL_17304059,RVQ--PTESIVRFPN-ITNLCP-------------FHEV-FNAT--...,Omicron,XBB.6,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
291537,EPI_ISL_17690414,RVQ--PTESIVRFPN-ITNLCP-------------FDEV-FNAT--...,Omicron,BA.2.9,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."


Now for each random seed for ADO, we grab the indices and apply them to this df. We can then compare them to the other embedding methods.

In [20]:
import os
import pandas as pd

data_dir = "/panfs/biopan03/prime_ml/prime/data/rbd/old"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/rbd/parquets"

parquet_file = os.path.join(parquet_dir, version_dir, "RBD.ADO.MSA_one_hot-embedded.parquet")
embedded_ado_df = pd.read_parquet(parquet_file, engine='fastparquet')

# Random sampling
min_sample_size = embedded_ado_df['variant'].value_counts().min()
sample_sizes = {"Alpha": min_sample_size,
                "Delta": min_sample_size,
                "Omicron": min_sample_size}

random_seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8]
for rnd_seed in random_seeds:
    seeded_sampled_dfs = [embedded_ado_df[embedded_ado_df['variant'] == variant].sample(n=sample_sizes[variant], random_state=rnd_seed)
                            for variant in sample_sizes.keys()]
        
    seeded_sampled_ado_df = pd.concat(seeded_sampled_dfs)
    save_as = f"{version_prefix}.sampled_seed{rnd_seed}.parquet"
    seeded_sampled_ado_df.to_parquet(os.path.join(parquet_dir, version_dir, save_as), index=False)

Parameter sweep.

In [2]:
import os
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden to 1 by setting random_state")

from sklearn.model_selection import ParameterGrid
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

import cupy as cp
from cuml.manifold import TSNE
from cuml.cluster import HDBSCAN
from cuml.preprocessing import LabelEncoder
from cuml.metrics.cluster import adjusted_rand_score, silhouette_score

def tsne_hdbscan_parameter_sweep(parquet_file, embedded_df, perplexity_list, min_samples_list, min_cluster_size_list):
    hdbscan_grid = {
        "min_samples": min_samples_list,
        "min_cluster_size": min_cluster_size_list,
    }
    hdbscan_param_list = list(ParameterGrid(hdbscan_grid))

    all_results = []

    # CPU -> GPU
    embedding_matrix = np.vstack(embedded_df["embedding"].values).astype(np.float32)
    X_gpu = cp.asarray(embedding_matrix)
    y_true = LabelEncoder().fit_transform(embedded_df["variant"].values)
    y_true_gpu = cp.asarray(y_true)

    n_samples = X_gpu.shape[0]
    early_exaggeration = 24.0 if n_samples > 10000 else 12.0 # Default 12.0
    learning_rate = max(n_samples / early_exaggeration, 200) # Default: 200

    # Parameter Sweep (TSNE)
    for perp in tqdm(perplexity_list, desc="TSNE sweep"):
        save_as = parquet_file.replace(".parquet", f".tsne-perplexity{perp}.parquet")

        if os.path.exists(save_as):
            print(f"Found existing t-SNE file, loading: {save_as}")
            tsne_df = pd.read_parquet(save_as, engine="fastparquet")

            X_red_cpu = tsne_df[["t-SNE component 1", "t-SNE component 2"]].values.astype(np.float32)
            X_red = cp.asarray(X_red_cpu)

        else:
            reducer = TSNE(
                n_components=2,
                perplexity=perp,
                n_neighbors=3 * perp,   # recommended by cuml, caps at 1023
                init="pca",             # like openTSNE
                metric="cosine",
                method="fft",
                random_state=42,
                early_exaggeration=early_exaggeration,
                learning_rate=learning_rate,
                learning_rate_method=None,
            )
            X_red = reducer.fit_transform(X_gpu)
            X_red_cpu = cp.asnumpy(X_red)

            # Save 
            tsne_df = pd.DataFrame({
                't-SNE component 1': X_red_cpu[:, 0],
                't-SNE component 2': X_red_cpu[:, 1],
                'Seq ID': embedded_df['seq_id'].values,
                'Variant': embedded_df['variant'].values,
                'Pango lineage': embedded_df['pango lineage'].values,
            })
            
            tsne_df.to_parquet(save_as, engine='fastparquet')

        # Parameter Sweep (HDBSCAN)
        for hdbscan_params in tqdm(hdbscan_param_list, leave=False):
            clusterer = HDBSCAN(
                min_samples=hdbscan_params["min_samples"],
                min_cluster_size=hdbscan_params["min_cluster_size"],
            )

            labels = clusterer.fit_predict(X_red)
            ari = float(adjusted_rand_score(y_true_gpu, labels))
            noise_fraction = float(cp.mean(labels == -1))

            non_noise_mask = labels != -1
            valid_cluster_labels = labels[non_noise_mask]
            
            n_non_noise_samples = int(cp.sum(non_noise_mask))
            n_non_noise_clusters = int(cp.unique(valid_cluster_labels).size)

            # silhouette needs at least two clusters AND
            # avoids the case where every non-noise point is its own cluster
            if n_non_noise_clusters > 1 and n_non_noise_samples > n_non_noise_clusters:
                silhouette_avg = float(silhouette_score(X_red[non_noise_mask], valid_cluster_labels,  metric="cosine"))
            else:
                silhouette_avg = np.nan                

            all_results.append({
                "perplexity": perp,
                "min_samples": hdbscan_params["min_samples"],
                "min_cluster_size": hdbscan_params["min_cluster_size"],
                "ari": ari,
                "noise_fraction": noise_fraction,
                "silhouette": silhouette_avg,
                "n_clusters": n_non_noise_clusters,
            })
            
        # cleanup
        del X_red
        cp._default_memory_pool.free_all_blocks()

    # Final results of both parameter sweeps
    full_results_df = pd.DataFrame(all_results) # All results
    return full_results_df

In [ ]:
%%time

sweep_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/rbd/param_sweeps"
parquet_dir = "/panfs/biopan03/prime_ml/prime/notebooks/clustering/rbd/parquets"

perplexity_list = [30, 50, 100, 250, 500, 600, 650, 700, 750, 800, 900, 1000]

hdbscan_grid = {
    "min_samples": [10, 25, 50, 100, 250, 500, 1000], # cuml HDBSCAN blocks min_samples above 1023
    "min_cluster_size": [50, 100, 250, 500, 1000, 2500, 5000, 10000, 15000, 20000, 22000],
}

random_seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8]
for rnd_seed in random_seeds:
    parquet_file = os.path.join(parquet_dir, version_dir, f"{version_prefix}.sampled_seed{rnd_seed}.parquet")
    embedded_df = pd.read_parquet(parquet_file, engine="fastparquet")

    seeded_full_results_tsne_df = tsne_hdbscan_parameter_sweep(
        parquet_file,
        embedded_df, 
        perplexity_list,
        hdbscan_grid["min_samples"],
        hdbscan_grid["min_cluster_size"],
    )

    seeded_full_results_tsne_df.to_csv(os.path.join(sweep_dir, version_dir, f"{version_prefix}.sampled_seed{rnd_seed}-tsne_hdbscan-parameter_sweep.csv"), index=False)

We now have our parameter sweeps for our MSA. Time to explore some plotting.